# Análise do Fator Casa na Premier League — TCC 2

**Aluno:** Matheus Ferreira Alphonse dos Anjos
**Orientador:** Prof. Francisco Pereira Junior (Thesko)
**Banca — revisão do TCC 1:** Prof. Cleber Gimenez Corrêa

Recorte: 11 temporadas da Premier League (2015/16 a 2025/26), 4.180 partidas.
Metodologia CRISP-DM.

---

## O que mudou desde o TCC 1

O TCC 1 executou as fases de Entendimento do Negócio, Entendimento dos Dados e
Preparação. Esta etapa faz três coisas:

1. **Corrige erros** identificados na revisão do código, dois dos quais alteram
   conclusões da monografia.
2. **Responde às análises pedidas pela banca**, com destaque para o
   comportamento do fator casa por posição final na tabela.
3. **Executa as fases de Modelagem e Avaliação**, com um modelo estatístico de
   Poisson bivariado e modelos preditivos com validação temporal.

> **Aviso ao leitor:** duas conclusões do TCC 1 não se sustentaram após a
> correção dos cálculos — a comparação entre o *Big Six* e os demais clubes, e a
> atribuição da queda do fator casa em 2020/21 à ausência de público. Ambas estão
> documentadas nas seções 3.1 e 5.2, com os números antigos e os novos lado a
> lado. Optou-se por explicitar a mudança em vez de apresentar apenas o
> resultado final.

---

## Como este notebook está organizado

A lógica da análise não está no notebook: está em módulos Python versionados
(`src/tcc/`), cobertos por testes automatizados. O notebook é a narrativa que
chama esses módulos. Assim, cada número apresentado aqui é gerado por código
testado, e não copiado à mão de uma execução anterior.

| Módulo | Responsabilidade |
| :--- | :--- |
| `coleta` | Download com cache local, manifesto e verificação de integridade |
| `preparacao` | Limpeza, variáveis derivadas e tabela de classificação |
| `estatistica` | Testes de hipótese, intervalos de confiança e tamanhos de efeito |
| `analise` | Fator casa por clube, faixa de posição, temporada e regime de público |
| `modelos` | Poisson bivariado (Dixon-Coles) com termo de mando de campo |
| `preditivo` | Features pré-jogo, baselines e validação temporal |
| `visualizacao` | Tema único de figuras e exportação para o LaTeX |


---
# 0. Preparação do ambiente

Executar esta célula primeiro. Ela clona o repositório e instala as bibliotecas.


In [ ]:
# Repositório do trabalho. Após a integração na branch principal, troque
# BRANCH para "main".
REPO = "https://github.com/matheustm29/TCC2.git"
BRANCH = "claude/tcc2-football-data-review-dgaeaw"

import os, subprocess, sys

# Idempotente: reexecutar esta célula não clona de novo nem aninha diretórios.
if not os.path.isdir("src/tcc"):
    if not os.path.isdir("TCC2"):
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO],
                       check=True)
    os.chdir("TCC2")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas>=2.2", "numpy>=1.26", "scipy>=1.11",
                "scikit-learn>=1.4", "matplotlib>=3.8"], check=True)

CAMINHO_SRC = os.path.join(os.getcwd(), "src")
if CAMINHO_SRC not in sys.path:
    sys.path.insert(0, CAMINHO_SRC)

print("Diretório de trabalho:", os.getcwd())
assert os.path.isdir("src/tcc"), "Repositório não encontrado — reexecute a célula."

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

from tcc import analise, coleta, config, estatistica, modelos, preditivo, preparacao, visualizacao

visualizacao.aplicar_tema()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("Temporadas do recorte:", ", ".join(config.TEMPORADAS))

---
# 1. Rastreabilidade: as anotações da banca

Cada anotação do parecer do TCC 1 e onde ela é tratada.


In [ ]:
anotacoes = pd.DataFrame([
    ("p. 3",  "Retirar as citações do resumo",                          "Redação",  "Pendente — texto"),
    ("p. 9",  "Trocar a referência da Wikipédia por livro ou artigo",   "Redação",  "Pendente — texto"),
    ("p. 10", "Utilizará Aprendizado de Máquina?",                      "Análise",  "Seções 5 e 6"),
    ("p. 12", "Primeiros colocados se comportam como os últimos?",      "Análise",  "Seção 4.1"),
    ("p. 13", "Ver o comportamento das equipes em cada temporada",      "Análise",  "Seção 4.2"),
    ("p. 13", "Ou grupos, os melhores e os piores times",               "Análise",  "Seção 4.1"),
    ("p. 13", "Média de outros dados: chutes, escanteios",              "Análise",  "Seção 4.3"),
    ("p. 13", "Melhorar a figura (colocar cores)",                      "Figuras",  "Todas as figuras"),
    ("p. 14", "Welch: usa quando as variâncias são diferentes",         "Método",   "Seção 3.2"),
], columns=["Página", "Anotação", "Natureza", "Onde é tratada"])

display(anotacoes.style.hide(axis="index"))

---
# 2. Fase 2 revisitada: dados e reprodutibilidade

O TCC 1 lia os CSVs direto da URL a cada execução, dentro de um `try/except` que
apenas imprimia a mensagem em caso de falha. Isso trazia dois riscos:

* o portal `football-data.co.uk` reescreve os arquivos (correções de placar,
  novas colunas), então duas execuções em datas diferentes podiam produzir
  números diferentes sem qualquer aviso;
* uma falha de rede geraria um conjunto de dados incompleto e a análise seguiria
  normalmente, com resultados silenciosamente errados.

Agora o download acontece uma vez, cada arquivo tem o SHA-256 registrado em um
manifesto, e qualquer inconsistência interrompe a execução.


In [ ]:
bruto = coleta.obter_bruto(config.TEMPORADAS)

print(f"Partidas carregadas: {len(bruto)}")
print(f"Temporadas: {bruto['Season'].nunique()}")
print(f"Partidas por temporada: {sorted(bruto.groupby('Season').size().unique())}")

In [ ]:
# As odds de mercado existem apenas na fonte primária. Quando disponíveis, entram
# como baseline do modelo preditivo (seção 6).
df = preparacao.preparar(bruto, colunas_extras=config.COLUNAS_ODDS)
longo = preparacao.formato_longo(df)

diagnostico = preditivo.diagnostico_odds(df)
print("Odds de mercado disponíveis:", diagnostico["disponivel"], "—", diagnostico["motivo"])
display(df.head())

## 2.1. Uma quebra de série na fonte

Antes de qualquer comparação histórica, vale verificar se as próprias variáveis
são comparáveis ao longo do tempo. A rotina abaixo sinaliza saltos abruptos entre
temporadas consecutivas — o tipo de variação que quase sempre indica mudança de
critério do provedor, e não mudança no futebol.

Aplicada a um recorte estendido (2009/10 em diante), ela encontra o seguinte.


In [ ]:
bruto_estendido = coleta.obter_bruto(config.TEMPORADAS_ESTENDIDAS)
longo_estendido = preparacao.formato_longo(preparacao.preparar(bruto_estendido))

quebras = analise.verificar_quebras_de_serie(longo_estendido)
display(quebras.round(3))

medias = longo_estendido.groupby("Season")[["chutes", "chutes_alvo", "gols_pro"]].mean()
display(medias.round(2).T)

A média de chutes no alvo da liga cai cerca de **37% entre 2012/13 e 2013/14**,
enquanto gols e chutes totais permanecem estáveis. Não é uma mudança no jogo: é
mudança de critério do provedor, que antes contabilizava finalizações bloqueadas
como chutes no alvo.

**Consequência prática:** qualquer comparação de chutes no alvo que atravesse
2013/14 é inválida. O recorte principal deste trabalho (2015/16 em diante) está
inteiramente depois da quebra e **não é afetado**; o cuidado importa apenas no
estudo de caso da seção 4.4, que usa dados anteriores.


---
# 3. Correções: antes e depois

Três correções alteram números publicados no TCC 1.


## 3.1. A comparação entre o *Big Six* e os demais clubes

**O que havia no TCC 1.** O cálculo agregava, para cada partida, os pontos do
mandante e os pontos do visitante:

```python
resumo_bigsix = df_clean.groupby('Home_BigSix').agg(
    Media_Pts_Casa=('PHT', 'mean'),
    Media_Pts_Fora=('PAT', 'mean'),   # pontos do adversário, não do grupo
)
```

Como `PAT` são os pontos do time visitante *daquela mesma partida*, a coluna
"pontos fora" não media o desempenho do Big Six como visitante — media o
desempenho dos adversários que **visitavam** o Big Six.

Dois sinais de que o número estava errado, ambos visíveis na própria saída do
TCC 1:

* o resultado atribuía **0,701 ponto por jogo** ao Big Six fora de casa, quando a
  tabela por clube do mesmo notebook mostrava Man City 2,05 e Liverpool 1,82;
* a linha dos demais clubes indicava desempenho **melhor fora** (1,409) do que em
  casa (1,339), o que contradiz toda a premissa do trabalho.

**A correção** calcula o diferencial por clube — cada linha já é uma observação de
um time, com o mando explícito — e só então compara os grupos. A unidade de
análise passa a ser o clube, e não a partida.


In [ ]:
por_time = analise.fator_casa_por_time(longo)
resultado_bigsix = analise.comparar_big_six(por_time)

comparacao = pd.DataFrame({
    "TCC 1 (com o erro)": ["+1,397", "−0,070", "—", "—", "< 0,001", "1.254 partidas"],
    "Corrigido": [
        f"{resultado_bigsix.detalhes['media_big_six']:+.3f}".replace(".", ","),
        f"{resultado_bigsix.detalhes['media_demais']:+.3f}".replace(".", ","),
        f"{resultado_bigsix.estimativa:+.3f}".replace(".", ","),
        f"[{resultado_bigsix.ic_inferior:+.3f}; {resultado_bigsix.ic_superior:+.3f}]".replace(".", ","),
        f"{resultado_bigsix.p_valor:.3f}".replace(".", ","),
        f"{resultado_bigsix.detalhes['n_big_six']} + {resultado_bigsix.detalhes['n_demais']} clubes",
    ],
}, index=["Diferencial Big Six", "Diferencial demais clubes",
          "Diferença entre grupos", "IC 95% da diferença", "p-valor", "n"])

display(comparacao)
print("Diferença estatisticamente significativa?", resultado_bigsix.significativo)

**Conclusão revisada.** A diferença entre os grupos não é estatisticamente
significativa, e o sinal é o oposto do afirmado no TCC 1: o Big Six apresenta
diferencial ligeiramente **maior**, não menor.

A afirmação *"Big Six: maior qualidade, menor dependência"* precisa ser
substituída por *"não há evidência de diferença entre os grupos"*.

Observe também que o `n` caiu de 1.254 para 28. Esse é o ponto: o teste original
parecia poderoso porque contava partidas, mas a hipótese era sobre **clubes**.


## 3.2. Os testes de significância

A banca anotou, ao lado do teste t de Welch: *"Usa quando as variâncias dos dois
grupos são diferentes"*. A observação aponta para um problema mais profundo do que
a escolha entre Welch e Student.

**Problema 1 — o teste t.** As séries comparadas eram os pontos do mandante e os
pontos do visitante da **mesma partida**. Elas são determinísticas entre si
(H → 3/0, D → 1/1, A → 0/3) e têm correlação de −0,95. A hipótese de
independência, que o `ttest_ind` exige, não se sustenta — e `equal_var=False`
corrige variâncias desiguais, não dependência entre amostras.

**Problema 2 — o qui-quadrado.** A hipótese nula usada foi a distribuição uniforme
(1/3 para vitória em casa, empate e vitória fora). Mas nenhuma teoria do fator
casa prevê 33% de empates: a taxa histórica é de cerca de 25%. A célula abaixo
demonstra o problema construindo um cenário **sem fator casa nenhum**.


In [ ]:
from scipy import stats

# Cenário contrafactual: mesma taxa real de empates, mas vitórias em casa e fora
# iguais — ou seja, fator casa inexistente.
sem_fator_casa = pd.Series(["H"] * 1595 + ["D"] * 990 + ["A"] * 1595)

qui2_uniforme, p_uniforme = stats.chisquare(
    [1595, 990, 1595], [len(sem_fator_casa) / 3] * 3
)
resultado_corrigido = estatistica.distribuicao_resultados_qui2(sem_fator_casa)

print("Cenário construído SEM fator casa:")
print(f"  H0 uniforme (usada no TCC 1) : p = {p_uniforme:.2e}  -> rejeita")
print(f"  H0 corrigida                 : p = {resultado_corrigido.p_valor:.3f}  -> não rejeita")
print()
print("A nula uniforme rejeitaria a hipótese mesmo num mundo sem vantagem do")
print("mandante. Ela mede 'a distribuição não é uniforme', não 'o mandante leva")
print("vantagem'.")

In [ ]:
# Testes corrigidos sobre os dados reais.
resultados = [
    estatistica.vantagem_mandante_binomial(df["FTR"]),
    estatistica.distribuicao_resultados_qui2(df["FTR"]),
    estatistica.diferenca_pontos_pareada(df["pts_mandante"], df["pts_visitante"]),
]

tabela_testes = estatistica.tabela_resultados(resultados)
display(tabela_testes.round(4))

**A conclusão central do TCC 1 sobrevive**: a vantagem do mandante é real e
altamente significativa sob os testes apropriados.

O que muda é a **magnitude**. O teste pareado indica uma diferença de 0,37 ponto
por jogo, com **d de Cohen de 0,14 — um efeito pequeno**. A monografia descreve o
fenômeno como *"assimetria estatística contundente"*. Com n = 4.180, um p-valor
minúsculo é barato; o que informa sobre a relevância prática é o tamanho de
efeito, ausente na versão anterior.


## 3.3. A taxa de conversão

O TCC 1 calculava a média das razões `gols / chutes no alvo` por partida e
excluía as partidas sem chutes no alvo. Ambos os passos distorcem o resultado:

* a média de razões dá o mesmo peso a um jogo com 1 chute no alvo e a um com 12 —
  a taxa de conversão do período é a **razão das somas**;
* o filtro removia 5,3% das partidas, e não aleatoriamente: removia justamente
  aquelas de conversão zero, inflando as duas médias.


In [ ]:
conversao = analise.taxa_conversao(df)
display(conversao.round(4))

print("TCC 1 (média de razões, com filtro): mandante 0,3327 | visitante 0,3212")
print(f"Corrigido (razão de somas)        : mandante {conversao.loc['casa', 'taxa_conversao']:.4f}"
      f" | visitante {conversao.loc['fora', 'taxa_conversao']:.4f}")

A conclusão qualitativa se mantém e agora vem com intervalo de confiança: as
faixas quase se tocam. Isso reforça a tese do TCC 1 de que **o fator casa é
volume de oportunidades, não eficiência de finalização**.


---
# 4. As análises pedidas pela banca

## 4.1. Primeiros colocados contra últimos colocados

> *"Os times que terminaram a temporada nas primeiras posições possuem o mesmo
> comportamento considerando o Fator Casa dos times que terminaram nas últimas
> posições."* — parecer, p. 12
>
> *"Ou grupos, os melhores e os piores times."* — parecer, p. 13

Responder a isso exige uma tabela de classificação, que o TCC 1 não construía. O
`Big Six` usado antes é um rótulo **fixo**, e a própria seção de limitações do
TCC 1 reconhecia que o grupo não é homogêneo ao longo do período (o Manchester
United declinou muito no recorte).

Aqui a faixa é recalculada **a cada temporada**, a partir da posição final de
fato: G6 (1º a 6º), Meio (7º a 14º) e Z6 (15º a 20º).


In [ ]:
tabela = preparacao.tabela_classificacao(df)
longo_com_posicao = preparacao.anexar_posicao(longo, tabela)

print("Classificação final de 2024/25:")
display(tabela[tabela["Season"] == "2425"].head(20).style.hide(axis="index"))

In [ ]:
por_faixa = analise.fator_casa_por_faixa(longo_com_posicao)
display(por_faixa[["jogos_casa", "pontos_casa", "pontos_fora", "dif_pontos",
                   "dif_gols_pro", "dif_chutes_alvo", "dif_escanteios"]].round(3))

visualizacao.fator_casa_por_faixa(por_faixa)

**Resposta.** Não, o comportamento não é o mesmo — e a diferença aponta na
direção contrária à intuição do TCC 1. O gradiente é monotônico e aparece em
todas as métricas: pontos, gols, chutes no alvo e escanteios. **Os times que
terminam nas primeiras posições ganham mais com o mando de campo do que os que
terminam nas últimas.**

O achado é ainda mais forte do que os números sugerem, por causa de um efeito
teto. O G6 já conquista cerca de 1,73 ponto por jogo como visitante, restando-lhe
apenas 1,27 ponto de margem até o máximo de 3. O Z6 conquista 0,72 fora e tem
2,28 de margem. Mesmo dispondo de muito menos espaço para crescer, o G6 cresce
mais.

Por isso a tabela por clube inclui a coluna `ganho_relativo`: a fração do espaço
ainda disponível que o clube converte jogando em casa — uma métrica sem teto.


## 4.2. O comportamento de cada equipe, temporada a temporada

> *"Ver o comportamento das equipes em cada temporada."* — parecer, p. 13


In [ ]:
por_time_temporada = analise.dif_por_time_temporada(longo_com_posicao)

print("Maiores diferenciais casa−fora em uma única temporada:")
display(
    por_time_temporada
    .sort_values("dif_pontos", ascending=False)
    .head(10)[["Season", "time", "posicao", "faixa", "pontos_casa", "pontos_fora", "dif_pontos"]]
    .round(3)
    .style.hide(axis="index")
)

In [ ]:
# Visão por clube ao longo do recorte, com o número de jogos explicitado.
colunas = ["jogos_casa", "n_temporadas", "amostra_suficiente", "big_six",
           "pontos_casa", "pontos_fora", "dif_pontos", "ganho_relativo"]

print("Maiores diferenciais no recorte completo:")
display(por_time.head(10)[colunas].round(3))

**Uma observação sobre amostra.** O ranking do TCC 1 era liderado pelo Hull City,
que disputou **uma única temporada** do recorte (19 jogos em casa), à frente de
clubes com 11 temporadas (209 jogos). A tabela não trazia o número de jogos, então
nada sinalizava a diferença.

A coluna `amostra_suficiente` marca clubes com menos de três temporadas. Eles não
são removidos — a limitação fica visível em vez de escondida —, mas são excluídos
dos testes estatísticos.


In [ ]:
visualizacao.dispersao_casa_fora(por_time)

## 4.3. Chutes e escanteios ao longo do tempo

> *"Média de outros dados: chutes, escanteios."* — parecer, p. 13

O TCC 1 acompanhava a evolução temporal apenas de pontos e gols; chutes e
escanteios apareciam somente agregados no período inteiro.

As duas métricas têm escalas diferentes e por isso ficam em painéis separados —
nunca em eixo duplo, que distorce a comparação visual.


In [ ]:
evolucao = analise.evolucao_temporal(longo, janela=3)

display(evolucao[["Season", "dif_pontos", "dif_pontos_mm3", "dif_chutes_alvo",
                  "dif_escanteios", "dif_amarelos"]].round(3).style.hide(axis="index"))

visualizacao.volume_ofensivo_por_temporada(evolucao)

Nota sobre a média móvel: o TCC 1 documentava no texto uma janela de três
temporadas, mas o código executava `rolling(window=2)`. Aqui a janela é um
parâmetro explícito da função, o que impede texto e código de divergirem.


In [ ]:
visualizacao.evolucao_pontos(evolucao)

In [ ]:
visualizacao.diferencial_por_temporada(evolucao)

## 4.4. Estudo de caso: mudança de estádio

A seção 4.5 do TCC 1 descrevia este estudo mas não continha código. Ela também era
inviável no recorte de 2015/16 em diante: o West Ham tinha uma única temporada no
Upton Park e o Brentford sequer havia disputado a Premier League antes da mudança
para o Gtech.

Com o recorte estendido a 2009/10, West Ham e Tottenham ganham linha de base. O
Brentford permanece impossível e por isso não entra na tabela.

Lembrando a seção 2.1: as métricas de finalização são anuladas quando a comparação
atravessa a quebra de série de 2013/14. Sem esse cuidado, o West Ham apareceria
"perdendo" quase dois chutes no alvo por jogo ao mudar de estádio — efeito que
seria inteiramente da planilha, não do futebol.


In [ ]:
estadios = analise.estudo_mudanca_estadio(longo_estendido)
display(estadios.round(3))

O West Ham perdeu cerca de metade da vantagem de mando ao deixar o Upton Park;
o Tottenham praticamente não mudou. É um resultado sugestivo e coerente com a
hipótese levantada no TCC 1 sobre a atmosfera das arenas modernas.

**Deve ser apresentado como estudo de caso exploratório, não como evidência
causal:** são dois clubes, e o período posterior à mudança também traz alterações
de elenco, de comissão técnica e de patamar competitivo que não foram controladas.


---
# 5. Fase 4: Modelagem

> *"Utilizará Aprendizado de Máquina?"* — parecer, p. 10

Sim. Esta seção e a seguinte executam as fases de Modelagem e Avaliação do
CRISP-DM, que o TCC 1 havia deixado para o TCC 2.

## 5.1. Por que um modelo, e não mais uma média

Todas as métricas do TCC 1 são médias condicionais. Nenhuma controla **contra
quem** cada clube jogou.

O "fator casa do Newcastle" (+0,568 ponto por jogo) mistura duas coisas que sem
modelo são indistinguíveis: a vantagem real do St. James' Park e o calendário que
o Newcastle enfrentou. Se por acaso ele recebeu adversários mais fracos do que
visitou, parte desse diferencial é sorteio de tabela.

O modelo de Poisson bivariado resolve isso. Para uma partida entre o mandante *i*
e o visitante *j*, os gols são modelados como Poisson com médias:

$$\lambda_{casa} = \exp(\mu + ataque_i - defesa_j + \gamma)$$
$$\lambda_{fora} = \exp(\mu + ataque_j - defesa_i)$$

Cada clube recebe um parâmetro de ataque e um de defesa; o mando de campo entra
como o parâmetro **compartilhado** $\gamma$. Como $\gamma$ é estimado junto com
a força de todos os clubes, ele mede a vantagem do mandante **já descontada** a
qualidade dos adversários. A quantidade $e^{\gamma} - 1$ é o acréscimo percentual
de gols atribuível ao mando.

A correção $\tau$ de Dixon e Coles (1997) ajusta a dependência nos placares
baixos (0-0, 1-0, 0-1, 1-1), onde o Poisson independente subestima empates.

> DIXON, M. J.; COLES, S. G. Modelling association football scores and
> inefficiencies in the football betting market. *Journal of the Royal
> Statistical Society: Series C*, v. 46, n. 2, p. 265–280, 1997.


In [ ]:
modelo = modelos.ajustar_poisson(df)
inferior, superior = modelo.ic_gamma

print(f"gamma                        = {modelo.gamma:.4f}")
print(f"IC 95%                       = [{inferior:.4f}; {superior:.4f}]")
print(f"Acréscimo de gols pelo mando = {modelo.acrescimo_percentual * 100:.1f}%")
print(f"rho (correção Dixon-Coles)   = {modelo.rho:.4f}")
print(f"Log-verossimilhança          = {modelo.log_verossimilhanca:.1f}")
print(f"Partidas                     = {modelo.n_partidas}")

**Jogar em casa vale cerca de 22% a mais de gols**, controlando pela força do
adversário. O $\rho$ negativo confirma o que a literatura descreve: o Poisson
independente subestima empates de placar baixo.

Uma checagem de sanidade: as forças estimadas devem ordenar os clubes de forma
reconhecível para quem acompanha a liga.


In [ ]:
forcas = modelo.tabela_forcas()
print("Cinco mais fortes:")
display(forcas.head(5).round(3))
print("Cinco mais fracos:")
display(forcas.tail(5).round(3))

In [ ]:
# O modelo produz probabilidades para qualquer confronto, nos dois mandos.
for mandante, visitante in [("Man City", "Burnley"), ("Burnley", "Man City")]:
    p = modelo.probabilidades(mandante, visitante)
    print(f"{mandante:12s} x {visitante:12s} -> "
          f"casa {p['H']:.1%} | empate {p['D']:.1%} | fora {p['A']:.1%}")

### Validação do estimador

Um modelo estatístico precisa ser testado antes de ser usado. O procedimento
adotado é o de **recuperação de parâmetro**: geram-se partidas sintéticas a partir
do próprio modelo, com um $\gamma$ conhecido, e verifica-se se o ajuste o
recupera.

Em 40 simulações independentes com $\gamma = 0{,}25$, a média das estimativas foi
**0,2362** e a cobertura do intervalo de 95% foi de **97,5%**. O procedimento está
em `tests/test_modelos.py`.

Essa validação revelou um problema na primeira implementação. O erro padrão vinha
da curvatura da verossimilhança na direção de $\gamma$ com os demais parâmetros
congelados, o que **subestimava a incerteza em cerca de 25%** e produzia cobertura
de 83% em um intervalo nominal de 95%. A causa é a correlação entre $\gamma$ e
$\mu$, que entram ambos na média de gols do mandante.

A versão atual usa **verossimilhança perfilada**: a cada deslocamento de
$\gamma$, todos os outros parâmetros são reotimizados. Os intervalos apresentados
aqui são, por isso, mais largos do que os de uma implementação ingênua — e
corretos.


## 5.2. O fator casa ao longo do tempo, ajustado

Ajustando o modelo separadamente em cada temporada, obtém-se uma série do fator
casa que — ao contrário do diferencial bruto de pontos — não se confunde com a
distribuição de força dos elencos daquela temporada nem com o calendário. E vem
com incerteza, o que deixa claro quando uma oscilação é apenas ruído.


In [ ]:
por_temporada = modelos.mando_por_temporada(df)
display(por_temporada.round(4).style.hide(axis="index"))

visualizacao.mando_ajustado_por_temporada(por_temporada)

Duas temporadas têm intervalo contendo zero: **2020/21**, já conhecida, e
**2024/25** — que não teve nada de pandemia. O segundo caso é novo e não aparecia
com a mesma clareza no diferencial bruto de pontos.

A leitura honesta é que $\gamma$ **oscila bastante** e os intervalos se sobrepõem
quase todos. Falar em "tendência de queda" exige cautela: 2022/23 (0,291) está no
mesmo patamar de 2016/17 e 2017/18. O que os dados mostram é variabilidade alta em
torno de um patamar estável, com dois vales.


## 5.3. O público: o que o TCC 1 concluiu e o que os dados sustentam

**O que havia no TCC 1.** A conclusão 5.2 afirmava que o experimento natural da
pandemia *"isola o papel do público como variável causal central, acima de fatores
como tempo de viagem ou familiaridade com o campo"*.

Essa conclusão vinha de uma comparação **entre temporadas**: as quatro temporadas
pré-pandemia contra a de 2020/21. O problema é que 2020/21 mudou muito mais do que
o público — pré-temporada encurtada, calendário congestionado e cinco
substituições por partida.

Além disso, a classificação do público era feita por **temporada inteira**, o que
contradiz a própria seção 3.3.2 da monografia, que afirma um filtro por data.
Apenas 92 das 380 partidas de 2019/20 foram disputadas sem público.

**A correção.** A classificação passa a ser por data da partida, com três regimes.
As janelas de liberação parcial (dezembro de 2020 e maio de 2021) recebem rótulo
próprio: as autorizações variavam por clube e por semana, e supor presença ou
ausência de torcida nesses jogos seria inventar dado.


In [ ]:
print("Partidas por regime de público:")
print(df["publico"].value_counts().to_string())
print()

contraste = analise.contraste_publico(longo)
display(contraste.round(3))

visualizacao.contraste_publico(contraste)

A linha decisiva é o par **2019/20 com público (+0,438)** contra **2019/20 sem
público (+0,457)**: mesma temporada, mesmos elencos, mesmo calendário, mesmas
regras. A única diferença entre os dois grupos é a torcida — e a vantagem do
mandante não caiu.

A queda só aparece em 2020/21 (−0,119).

Repare ainda na coluna de cartões amarelos. Com público, o mandante recebe
**menos** amarelos que o visitante (−0,25 em 2019/20). Sem público, essa vantagem
desaparece (+0,03) — **e desaparece já em 2019/20, quando a vantagem em pontos não
caiu**. Os dois mecanismos se separam: a torcida parece influenciar a arbitragem,
mas o viés de arbitragem não é o que sustenta a vantagem em pontos.


### O teste formal, com o modelo

O contraste acima é descritivo. Para testá-lo controlando pela força dos
adversários, o mando é modelado como $\gamma + \delta \cdot \mathbb{1}[\text{sem
público}]$. Como $\delta$ é um único parâmetro extra, ele fica bem identificado
mesmo em recortes pequenos — ajustar o modelo inteiro separadamente em 92 partidas
não seria viável.


In [ ]:
df_com_indicadora = df.copy()
df_com_indicadora["sem_publico"] = (df_com_indicadora["publico"] == "sem").astype(int)

recortes = {
    "2019/20 (intratemporada)": df_com_indicadora[df_com_indicadora["Season"] == "1920"],
    "Recorte completo (19/20 + 20/21)": df_com_indicadora[df_com_indicadora["publico"] != "limitado"],
}

linhas = []
for rotulo, recorte in recortes.items():
    efeito = modelos.ajustar_efeito_no_mando(recorte, "sem_publico")
    inf, sup = efeito.ic_delta
    linhas.append({
        "Recorte": rotulo,
        "gamma (com público)": round(efeito.gamma_base, 4),
        "delta (sem público)": round(efeito.delta, 4),
        "IC 95%": f"[{inf:+.3f}; {sup:+.3f}]",
        "p-valor": round(efeito.p_valor, 4),
        "n sem público": efeito.n_com_condicao,
        "n com público": efeito.n_sem_condicao,
    })

display(pd.DataFrame(linhas).style.hide(axis="index"))

**Conclusão revisada.** Dentro de 2019/20, a ausência de torcida **não desloca o
mando de campo** (p = 0,68). O efeito agregado, esse sim significativo, vem
inteiramente de 2020/21 — temporada da qual saem 78% das partidas sem público.

É preciso ser preciso sobre o alcance disso. O intervalo de 2019/20 ainda contém o
valor estimado no agregado, de modo que os dados **não excluem** um efeito daquele
tamanho. Também não é possível construir o contraste dentro de 2020/21, porque
nenhuma partida daquela temporada teve público pleno.

O que se pode afirmar com segurança é: **a vantagem do mandante caiu a zero em
2020/21, e atribuir essa queda especificamente à ausência de torcida é uma
inferência que os dados de 2019/20 não corroboram.** A afirmação do TCC 1 de que o
experimento natural "isola" o papel do público é mais forte do que a evidência
permite.


---
# 6. Modelagem preditiva

## 6.1. O erro que este desenho evita

Este é o ponto mais delicado da etapa. As colunas `HS`, `HST`, `HC`, `HF` e as de
cartões — as mesmas que o TCC 1 usou na matriz de correlação — só existem
**depois** do apito final. Usá-las para prever o resultado da partida produziria
acurácia alta e um trabalho errado. É o erro mais comum em trabalhos de previsão
esportiva, e o mais fácil de não perceber.

As features usadas aqui empregam apenas informação disponível **antes do apito
inicial**:

| Feature | Descrição |
| :--- | :--- |
| `elo_casa`, `elo_fora`, `elo_diferenca` | Rating Elo antes da partida |
| `forma_pontos_*` | Pontos por jogo nas 5 partidas anteriores |
| `forma_gols_pro_*`, `forma_gols_contra_*` | Gols nas 5 partidas anteriores |
| `forma_casa_mandante`, `forma_fora_visitante` | Forma específica de mando |
| `descanso_casa`, `descanso_fora` | Dias desde a partida anterior do clube |
| `rodada` | Momento da temporada |
| `sem_publico` | Regime de público |

Isso não é apenas uma declaração de intenção — é verificado por teste automatizado.


In [ ]:
proibidas = {"HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF",
             "HY", "AY", "HR", "AR", "FTHG", "FTAG", "FTR"}

print("Features usadas:", len(preditivo.FEATURES))
for f in preditivo.FEATURES:
    print("  -", f)
print()
print("Alguma estatística pós-jogo entre as features?",
      bool(set(preditivo.FEATURES) & proibidas))

### A trava experimental contra vazamento

Listar as features não prova que não há vazamento — um erro de `shift` bastaria
para a média móvel incluir a própria partida. O teste em `tests/test_preditivo.py`
verifica isso experimentalmente:

1. altera o placar de uma partida no meio da base;
2. reconstrói **todas** as features do zero;
3. exige que **nenhuma feature daquela mesma partida** tenha mudado.

Um segundo teste faz a contraprova: exige que as features das partidas
**seguintes** tenham mudado. Sem ele, a primeira asserção passaria trivialmente
caso as features fossem constantes.

A célula abaixo reproduz o experimento.


In [ ]:
def construir(quadro):
    preparado = preparacao.preparar(quadro)
    return preditivo.construir_features(
        preparado, preparacao.formato_longo(preparado)
    ).sort_values(preditivo.ORDENACAO).reset_index(drop=True)

original = construir(bruto)

# Inverte o placar de uma partida no meio da base.
posicao = len(bruto) // 2
alterado = bruto.copy()
alterado.loc[posicao, ["FTHG", "FTAG"]] = [7, 0]
alterado.loc[posicao, "FTR"] = "H"
modificado = construir(alterado)

# A base bruta traz a data como texto e a preparada como datetime, então a
# chave precisa ser convertida antes do cruzamento.
chave = ["Date", "HomeTeam", "AwayTeam"]
alvo = alterado.loc[[posicao], chave].copy()
alvo["Date"] = preparacao.converter_datas(alvo["Date"])

antes = original.merge(alvo, on=chave)
depois = modificado.merge(alvo, on=chave)
assert len(antes) == 1 and len(depois) == 1

mudou = [f for f in preditivo.FEATURES
         if not np.isclose(antes.iloc[0][f], depois.iloc[0][f])]

print("Partida alterada:", antes.iloc[0]["HomeTeam"], "x", antes.iloc[0]["AwayTeam"])
print("Features da própria partida que mudaram:", mudou if mudou else "NENHUMA")
print()

# Contraprova: as partidas seguintes precisam mudar.
mascara_alvo = (
    (original["Date"] == alvo["Date"].iloc[0])
    & (original["HomeTeam"] == alvo["HomeTeam"].iloc[0])
    & (original["AwayTeam"] == alvo["AwayTeam"].iloc[0])
)
posicao_preparada = int(original.index[mascara_alvo][0])
posteriores = original.index > posicao_preparada
elo_mudou = not np.allclose(original.loc[posteriores, "elo_casa"],
                            modificado.loc[posteriores, "elo_casa"])
print("O Elo das partidas seguintes mudou (contraprova)?", elo_mudou)

## 6.2. Protocolo de validação

A validação é **temporal** (*walk-forward*), nunca aleatória. Um
`train_test_split` embaralharia futuro com passado e inflaria todas as métricas.

Para prever a temporada *t*, o treino é tudo o que aconteceu antes dela. Dentro da
temporada, o histórico cresce a cada 10 partidas e **todos** os modelos são
reajustados sobre ele. Esse detalhe importa: se o Dixon-Coles reestimasse
parâmetros ao longo da temporada e os classificadores não, a comparação mediria o
protocolo de reajuste em vez da qualidade dos modelos.

Temporadas de teste: 2018/19 a 2025/26. As três primeiras servem de treino
inicial.

### Modelos comparados

| Modelo | Papel |
| :--- | :--- |
| `frequencia_base` | Frequências H/D/A do histórico. É o **piso**: um modelo que não bate isto não aprendeu nada além da distribuição marginal. |
| `sempre_casa` | A heurística "o mandante vence", com a confiança que o histórico justifica. |
| `dixon_coles` | O modelo estatístico da seção 5, usado como preditor. |
| `regressao_logistica` | Regressão logística multinomial sobre as features. |
| `gradient_boosting` | *Gradient boosting* sobre as mesmas features. |
| `odds_mercado` | Probabilidades implícitas nas odds do Bet365, sem a margem da casa. Entra apenas quando a fonte traz as colunas. |

### Métricas

**Log-loss** e **Brier** são as principais, porque o problema é probabilístico.
A **acurácia** entra como secundária: ela não distingue um modelo bem calibrado de
um confiante e errado, e num problema em que 44% das partidas terminam em vitória
do mandante ela é enganosamente fácil de parecer boa.


### Resultados

A validação completa leva cerca de 20 a 30 minutos, porque reajusta cinco modelos
a cada 10 partidas ao longo de 8 temporadas. Para a apresentação, a célula abaixo
carrega os resultados já calculados e versionados no repositório. A célula
seguinte permite recalcular tudo do zero.


In [ ]:
resumo = pd.read_csv("tabelas/preditivo_resumo.csv", index_col=0)
metricas = pd.read_csv("tabelas/preditivo_metricas_por_temporada.csv")

print("Desempenho agregado nas 8 temporadas de teste:")
display(resumo.round(4))

visualizacao.desempenho_preditivo(resumo)

In [ ]:
# Desempenho temporada a temporada.
pivo = metricas.pivot(index="Season", columns="modelo", values="log_loss")
print("Log-loss por temporada (menor é melhor):")
display(pivo.round(4))

In [ ]:
calibracao = pd.read_csv("tabelas/preditivo_calibracao.csv")
visualizacao.calibracao(calibracao)

A curva de calibração responde a uma pergunta que a acurácia não alcança: entre
as partidas em que o modelo atribui 70% de chance de vitória ao mandante, o
mandante vence de fato cerca de 70% das vezes? Um modelo bem calibrado põe os
pontos sobre a diagonal.


### Recalcular do zero (opcional)

Executar apenas se houver tempo — a célula leva de 20 a 30 minutos.


In [ ]:
# RECALCULAR = True para reexecutar a validação temporal completa.
RECALCULAR = False

if RECALCULAR:
    features = preditivo.construir_features(df, longo)
    temporadas_teste = ("1819", "1920", "2021", "2122", "2223", "2324", "2425", "2526")
    metricas_novas, previsoes = preditivo.validacao_temporal(features, temporadas_teste)
    display(preditivo.resumo_por_modelo(metricas_novas).round(4))
else:
    print("Usando os resultados versionados em tabelas/.")
    print("Defina RECALCULAR = True para reexecutar (20 a 30 minutos).")

---
# 7. Considerações finais

## 7.1. Conclusões que se mantêm

* **O fator casa existe e é robusto.** Todos os testes corrigidos rejeitam a
  hipótese de simetria com folga. Jogar em casa vale cerca de **22% a mais de
  gols**, controlando pela força do adversário.
* **O fator casa é volume, não eficiência.** A taxa de conversão de chutes no alvo
  em gols é praticamente a mesma para mandantes (32,6%) e visitantes (31,8%). A
  vantagem vem do maior número de oportunidades criadas.

## 7.2. Conclusões que mudaram

* **O Big Six não depende menos do mando.** A diferença entre os grupos não é
  significativa (p = 0,35) e o sinal é oposto ao afirmado no TCC 1.
* **Times de cima têm mais fator casa, não menos.** O gradiente por posição final
  é monotônico: G6 0,450, meio 0,349, Z6 0,319 — e é mais forte do que parece,
  porque o efeito teto trabalha contra os primeiros colocados.
* **O público não explica a queda de 2020/21.** Dentro de 2019/20, com os mesmos
  elencos e o mesmo calendário, a ausência de torcida não desloca a vantagem do
  mandante. A evidência do efeito repousa inteiramente sobre uma temporada que
  mudou muito além do público.

## 7.3. Um achado novo

O viés de arbitragem e a vantagem em pontos **se separam**. Com público, o
mandante recebe menos cartões amarelos que o visitante; sem público, essa
vantagem desaparece — e desaparece já em 2019/20, quando a vantagem em pontos não
caiu. A torcida parece influenciar a arbitragem, mas o viés de arbitragem não é o
mecanismo que sustenta a vantagem em pontos.

## 7.4. Limitações

* **$\gamma$ é único para toda a liga.** O modelo não estima uma vantagem de
  mando por clube. Fazê-lo exigiria um parâmetro por clube, com amostra pequena e
  necessidade de encolhimento — extensão natural, fora do escopo atual.
* **Sem variáveis de contexto** que a literatura aponta: distância de viagem,
  horário da partida, árbitro designado e ocupação do estádio.
* **A classificação de público é por janela de datas.** As liberações parciais de
  dezembro de 2020 e maio de 2021 variavam por clube e por semana; essas partidas
  ficam em rótulo próprio e fora dos contrastes.
* **O estudo de caso dos estádios é exploratório.** Dois clubes, sem controle para
  mudanças de elenco e de comissão técnica no mesmo período.
* **Quebra de série na fonte.** Comparações de chutes no alvo que atravessem
  2013/14 são inválidas. O recorte principal não é afetado.

## 7.5. Reprodutibilidade

```bash
pip install -r requirements.txt
python scripts/executar_analise.py     # análise descritiva e inferencial
python scripts/executar_modelagem.py   # Dixon-Coles e modelos preditivos
pytest tests/ -q                       # bateria de testes
```

Cada correção descrita aqui tem um teste que a trava, incluindo um cenário
construído sem fator casa para garantir que o qui-quadrado corrigido não o acuse,
a recuperação de parâmetros conhecidos em dados simulados e a trava experimental
contra vazamento.
